In [1]:
import os
import random
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

In [2]:
project_dir = Path.cwd()
data_dir = project_dir / "data"

# Set to False to train on the full CICIDS2017 pool and True to run a faster subset during testing.
USE_SAMPLE = False
SAMPLE_FRAC = 0.25

csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in data/")
for path in csv_files:
    print(" -", path.name)

frames = []
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path, encoding="latin1", low_memory=False)
    except Exception:
        try:
            df = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
        except Exception as exc:
            print(f"Skipping {csv_path.name}: {exc}")
            continue

    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]

    label_col = next((col for col in df.columns if col.lower() == "label"), None)
    if df.empty or label_col is None:
        print(f"Skipping {csv_path.name}: no usable label column")
        continue

    if label_col != "Label":
        df = df.rename(columns={label_col: "Label"})

    frames.append(df)

if not frames:
    raise ValueError("No valid CICIDS2017 CSV files were found in the data folder.")

df = pd.concat(frames, ignore_index=True)
print("Merged dataset shape:", df.shape)

# Remove metadata columns that should not be used as predictors.
metadata_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]
metadata_cols = [col for col in metadata_cols if col in df.columns]
df = df.drop(columns=metadata_cols, errors="ignore")

df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)

for col in df.columns:
    if col == "Label":
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert all labels to binary: BENIGN=0, any attack=1.
df["Label"] = df["Label"].astype(str).str.strip()
df["Label"] = df["Label"].apply(lambda label: 0 if label == "BENIGN" else 1)
print("\nLabel distribution:")
print(df["Label"].value_counts().sort_index())

# Fill missing values with a safe per-column median.
for col in df.columns:
    if col == "Label":
        continue
    median = df[col].median()
    df[col] = df[col].fillna(median if pd.notna(median) else 0)

if USE_SAMPLE:
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED)
    print("\nSampled dataset shape:", df.shape)

X = df.drop(columns=["Label"])
y = df["Label"].astype(int)

# Stratified 60/20/20 split: train, validation, test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=SEED,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

print("\nTrain/Validation/Test sizes:")
print(len(X_train), len(X_val), len(X_test))


def compute_binary_metrics(y_true, y_pred, y_proba=None):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    far = fp / (fp + tn) if (fp + tn) else 0.0
    roc_auc = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "far": far,
        "roc_auc": roc_auc,
    }


models = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                    class_weight="balanced",
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "MLP": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    max_iter=500,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}

validation_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None
    validation_results[name] = compute_binary_metrics(y_val, val_pred, val_proba)

best_model_name = max(validation_results, key=lambda key: validation_results[key]["f1"])
print("\nBest validation model by F1:", best_model_name)
print(pd.DataFrame(validation_results).T.sort_values("f1", ascending=False))

# Final test-set evaluation for the chosen model.
test_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_pred = model.predict(X_test)
    test_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    test_results[name] = compute_binary_metrics(y_test, test_pred, test_proba)

summary = pd.DataFrame(test_results).T.sort_values("f1", ascending=False)
print("\nSummary of test-set model performance:")
print(summary)

# Ablation study: feature scaling for Logistic Regression.
logreg_noscale = LogisticRegression(
    max_iter=2000,
    random_state=SEED,
    class_weight="balanced",
)
logreg_noscale.fit(X_train, y_train)
logreg_no_scale_pred = logreg_noscale.predict(X_val)
logreg_no_scale_metrics = compute_binary_metrics(y_val, logreg_no_scale_pred)

logreg_scaled = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            random_state=SEED,
            class_weight="balanced",
        ),
    ),
])
logreg_scaled.fit(X_train, y_train)
logreg_scaled_pred = logreg_scaled.predict(X_val)
logreg_scaled_metrics = compute_binary_metrics(y_val, logreg_scaled_pred)

print("\nAblation: Logistic Regression with vs without scaling")
print("Without scaling:", logreg_no_scale_metrics)
print("With scaling:", logreg_scaled_metrics)

# Feature importance for the best tree-based model.
rf_model = models["Random Forest"]
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 features:")
print(feature_importance.head(10))

fig_dir = project_dir / "figures"
fig_dir.mkdir(exist_ok=True)
feature_importance.head(15).sort_values().plot(kind="barh", figsize=(10, 6), color="steelblue")
plt.title("Top 15 features for CICIDS2017 attack detection")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(fig_dir / "top_features.png", dpi=150)
plt.close()

results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)
summary.to_csv(results_dir / "model_metrics.csv")
print("\nSaved metrics to:", results_dir / "model_metrics.csv")
print("Saved feature plot to:", fig_dir / "top_features.png")

Found 8 CSV files in data/
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


Merged dataset shape: (3119345, 85)



Label distribution:
Label
0    2263765
1     550035
Name: count, dtype: int64



Train/Validation/Test sizes:
1688280 562760 562760



Best validation model by F1: Random Forest
                     accuracy  precision    recall        f1       far  \
Random Forest        0.999597   0.998619  0.999318  0.998969  0.000336   
MLP                  0.996839   0.992806  0.991010  0.991907  0.001745   
Logistic Regression  0.919253   0.717943  0.966720  0.823963  0.092280   

                      roc_auc  
Random Forest        0.999986  
MLP                  0.999891  
Logistic Regression  0.980145  



Summary of test-set model performance:
                     accuracy  precision    recall        f1       far  \
Random Forest        0.999606   0.998601  0.999382  0.998991  0.000340   
MLP                  0.996823   0.992518  0.991219  0.991868  0.001816   
Logistic Regression  0.919383   0.718359  0.966529  0.824167  0.092072   

                      roc_auc  
Random Forest        0.999987  
MLP                  0.999884  
Logistic Regression  0.979806  



Ablation: Logistic Regression with vs without scaling
Without scaling: {'accuracy': 0.8501083943421708, 'precision': 0.5956710473320703, 'recall': 0.7259901642622743, 'f1': 0.6544057095800164, 'far': np.float64(0.11973415968530303), 'roc_auc': nan}
With scaling: {'accuracy': 0.9192533229085222, 'precision': 0.7179428324534518, 'recall': 0.9667202996172971, 'f1': 0.8239628408611065, 'far': np.float64(0.09227989654403174), 'roc_auc': nan}

Top 10 features:
Destination Port           0.059765
Average Packet Size        0.050617
Packet Length Std          0.049739
Source Port                0.049586
Init_Win_bytes_forward     0.043955
Packet Length Variance     0.042901
Init_Win_bytes_backward    0.039863
Min Packet Length          0.034333
Max Packet Length          0.031962
Fwd Packet Length Min      0.031044
dtype: float64

Saved metrics to: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/Lab1/results/model_metrics.csv
Sav